# Product Return Analysis

## Project Overview
Analyze product returns by category, reason, customer type, and season. Focus on identifying high-return items, quantifying financial impact, and uncovering likely root causes to reduce return rates.

## Learning Objectives
- Compute return rates across multiple dimensions (product, category, season, customer type)
- Quantify the financial impact of returns
- Identify patterns and root causes behind high return rates
- Generate actionable recommendations to reduce returns

## Problem Statement
An e-commerce retailer is experiencing a high return rate that erodes margins. Understanding which products are returned most, why, and by whom is critical to reduce waste and improve customer satisfaction.

## Why This Project Matters
Product returns cost retailers billions annually — not just in refunds, but in shipping, restocking, and lost inventory value. Even a 1–2% reduction in return rate can significantly improve profitability.

## Dataset Overview
Synthetic order and return data with realistic patterns: ~10K orders, ~20% return rate. Includes product category, return reason, customer type, order date, and financials.

## Dataset Source and License Notes
- Synthetic data generated for educational purposes
- Patterns modeled on common e-commerce return behaviours
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n_orders = 10000

categories = ['Electronics', 'Clothing', 'Home & Kitchen', 'Sports', 'Books', 'Toys', 'Beauty']
cat_weights = [0.20, 0.25, 0.15, 0.10, 0.10, 0.10, 0.10]
# Return rate varies by category
cat_return_rates = {'Electronics': 0.18, 'Clothing': 0.30, 'Home & Kitchen': 0.12,
                    'Sports': 0.10, 'Books': 0.05, 'Toys': 0.15, 'Beauty': 0.22}

return_reasons = ['Defective', 'Wrong Size/Fit', 'Not as Described', 'Changed Mind',
                  'Arrived Late', 'Better Price Found', 'Wrong Item Shipped']
reason_weights_by_cat = {
    'Electronics': [0.30, 0.02, 0.25, 0.20, 0.10, 0.08, 0.05],
    'Clothing': [0.05, 0.45, 0.15, 0.20, 0.05, 0.05, 0.05],
    'Home & Kitchen': [0.20, 0.05, 0.30, 0.25, 0.10, 0.05, 0.05],
    'Sports': [0.15, 0.20, 0.20, 0.25, 0.10, 0.05, 0.05],
    'Books': [0.10, 0.02, 0.20, 0.40, 0.15, 0.08, 0.05],
    'Toys': [0.25, 0.03, 0.25, 0.25, 0.10, 0.07, 0.05],
    'Beauty': [0.15, 0.05, 0.30, 0.25, 0.10, 0.10, 0.05],
}

customer_types = ['New', 'Returning', 'VIP']
cust_weights = [0.35, 0.50, 0.15]

dates = pd.date_range('2023-01-01', '2024-12-31', freq='D')

cat_choices = np.random.choice(categories, size=n_orders, p=cat_weights)
order_dates = np.random.choice(dates, size=n_orders)
cust_types = np.random.choice(customer_types, size=n_orders, p=cust_weights)

prices = []
for cat in cat_choices:
    if cat == 'Electronics': prices.append(round(np.random.lognormal(4.5, 0.6), 2))
    elif cat == 'Clothing': prices.append(round(np.random.lognormal(3.5, 0.5), 2))
    elif cat == 'Books': prices.append(round(np.random.lognormal(2.8, 0.4), 2))
    else: prices.append(round(np.random.lognormal(3.2, 0.6), 2))

prices = np.clip(prices, 5, 2000)

# Determine returns
is_returned = []
ret_reasons = []
for i, cat in enumerate(cat_choices):
    rate = cat_return_rates[cat]
    # New customers return slightly more
    if cust_types[i] == 'New': rate *= 1.15
    elif cust_types[i] == 'VIP': rate *= 0.80
    returned = np.random.random() < rate
    is_returned.append(returned)
    if returned:
        ret_reasons.append(np.random.choice(return_reasons, p=reason_weights_by_cat[cat]))
    else:
        ret_reasons.append(None)

df = pd.DataFrame({
    'OrderID': [f'ORD{i:06d}' for i in range(n_orders)],
    'OrderDate': order_dates,
    'Category': cat_choices,
    'Price': prices,
    'CustomerType': cust_types,
    'Returned': is_returned,
    'ReturnReason': ret_reasons,
})
df['OrderDate'] = pd.to_datetime(df['OrderDate'])
df['Month'] = df['OrderDate'].dt.to_period('M')
df['Quarter'] = df['OrderDate'].dt.to_period('Q')
df['Season'] = df['OrderDate'].dt.month.map(
    {12:'Winter',1:'Winter',2:'Winter',3:'Spring',4:'Spring',5:'Spring',
     6:'Summer',7:'Summer',8:'Summer',9:'Fall',10:'Fall',11:'Fall'})

print(f'Dataset shape: {df.shape}')
print(f'Return rate: {df["Returned"].mean():.1%}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values:\n{df.isnull().sum()}')
print(f'\nReturned orders with no reason: {df[df["Returned"] & df["ReturnReason"].isna()].shape[0]}')
print(f'Non-returned orders with reason: {df[~df["Returned"] & df["ReturnReason"].notna()].shape[0]}')
print(f'Price range: ${df["Price"].min():.2f} - ${df["Price"].max():.2f}')
print(f'Date range: {df["OrderDate"].min().date()} to {df["OrderDate"].max().date()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Orders by category
df['Category'].value_counts().plot.bar(ax=axes[0,0], edgecolor='black')
axes[0,0].set_title('Orders by Category')
axes[0,0].tick_params(axis='x', rotation=45)

# Return rate by category
ret_by_cat = df.groupby('Category')['Returned'].mean().sort_values(ascending=False)
ret_by_cat.plot.bar(ax=axes[0,1], edgecolor='black', color='coral')
axes[0,1].set_title('Return Rate by Category')
axes[0,1].set_ylabel('Return Rate')
axes[0,1].tick_params(axis='x', rotation=45)

# Price distribution: returned vs not
df[df['Returned']]['Price'].hist(ax=axes[1,0], bins=40, alpha=0.6, label='Returned', edgecolor='black')
df[~df['Returned']]['Price'].hist(ax=axes[1,0], bins=40, alpha=0.6, label='Kept', edgecolor='black')
axes[1,0].set_title('Price Distribution: Returned vs Kept')
axes[1,0].legend()

# Monthly return rate
monthly_ret = df.groupby('Month')['Returned'].mean()
monthly_ret.plot(ax=axes[1,1], marker='o')
axes[1,1].set_title('Monthly Return Rate')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Target Analysis — Return Reasons

In [ ]:
returns = df[df['Returned']].copy()
print(f'Total returns: {len(returns)}')
print(f'\nReturn reasons:\n{returns["ReturnReason"].value_counts()}')

fig, ax = plt.subplots(figsize=(10, 5))
returns['ReturnReason'].value_counts().plot.barh(ax=ax, edgecolor='black', color='steelblue')
ax.set_title('Return Reasons Distribution')
ax.set_xlabel('Count')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'return_reasons.png'), dpi=100, bbox_inches='tight')
plt.show()

## Return Rate by Customer Type

In [ ]:
ret_by_cust = df.groupby('CustomerType').agg(
    orders=('Returned', 'count'),
    returns=('Returned', 'sum'),
    return_rate=('Returned', 'mean'),
    avg_price=('Price', 'mean'),
).round(3)
print(ret_by_cust)

fig, ax = plt.subplots(figsize=(8, 5))
ret_by_cust['return_rate'].plot.bar(ax=ax, edgecolor='black', color=['#ff9999','#66b3ff','#99ff99'])
ax.set_title('Return Rate by Customer Type')
ax.set_ylabel('Return Rate')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'return_by_customer.png'), dpi=100, bbox_inches='tight')
plt.show()

## Return Reasons by Category (Heatmap)

In [ ]:
reason_cat = pd.crosstab(returns['Category'], returns['ReturnReason'], normalize='index')
plt.figure(figsize=(12, 6))
sns.heatmap(reason_cat, annot=True, fmt='.2f', cmap='YlOrRd')
plt.title('Return Reason Distribution by Category (Row-Normalized)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'reason_by_category.png'), dpi=100, bbox_inches='tight')
plt.show()

## Seasonal Return Patterns

In [ ]:
season_ret = df.groupby('Season').agg(
    orders=('Returned', 'count'),
    returns=('Returned', 'sum'),
    return_rate=('Returned', 'mean'),
).round(3).reindex(['Winter', 'Spring', 'Summer', 'Fall'])
print(season_ret)

fig, ax = plt.subplots(figsize=(8, 5))
season_ret['return_rate'].plot.bar(ax=ax, edgecolor='black', color=['#4fc3f7','#81c784','#ffb74d','#e57373'])
ax.set_title('Return Rate by Season')
ax.set_ylabel('Return Rate')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'seasonal_returns.png'), dpi=100, bbox_inches='tight')
plt.show()

## Financial Impact Analysis

In [ ]:
total_revenue = df['Price'].sum()
returned_revenue = returns['Price'].sum()
return_cost_rate = 0.15  # estimated cost per return as fraction of price
estimated_return_cost = returned_revenue * return_cost_rate

print(f'Total revenue:        ${total_revenue:,.2f}')
print(f'Returned value:       ${returned_revenue:,.2f}')
print(f'Return % of revenue:  {returned_revenue/total_revenue:.1%}')
print(f'Est. return cost (15% of returned value): ${estimated_return_cost:,.2f}')

# Impact by category
impact = df.groupby('Category').agg(
    total_rev=('Price', 'sum'),
    returned_rev=('Price', lambda x: x[df.loc[x.index, 'Returned']].sum()),
    return_rate=('Returned', 'mean'),
).round(2)
impact['lost_pct'] = (impact['returned_rev'] / impact['total_rev'] * 100).round(1)
impact = impact.sort_values('returned_rev', ascending=False)
print('\nFinancial Impact by Category:')
print(impact)

## High-Return Products — Root Cause Hypotheses

In [ ]:
print('Root Cause Analysis by Category + Return Reason:')
print('=' * 60)
top_combos = returns.groupby(['Category', 'ReturnReason']).size().reset_index(name='Count')
top_combos = top_combos.sort_values('Count', ascending=False).head(10)
for _, row in top_combos.iterrows():
    print(f"  {row['Category']} — {row['ReturnReason']}: {row['Count']} returns")

print()
print('Hypothesized Root Causes:')
print('  - Clothing + Wrong Size/Fit: Inconsistent sizing charts')
print('  - Electronics + Defective: Quality control gaps or fragile shipping')
print('  - Electronics + Not as Described: Misleading product images/specs')
print('  - Beauty + Not as Described: Color/shade mismatch from online display')
print('  - Home & Kitchen + Not as Described: Dimensions unclear in listings')

## Interpretation and Business Insight
- **Clothing** has the highest return rate (~30%), driven primarily by sizing issues — a well-known industry challenge
- **Electronics** returns are costly due to high unit prices, even with a moderate return rate
- **New customers** return more frequently, suggesting onboarding and expectation management gaps
- **"Not as Described"** is a common reason across categories, indicating listing quality issues
- Financial impact is concentrated in Electronics and Clothing — these categories deserve priority attention

## Limitations
- Synthetic data; real return patterns are more complex and category-specific
- No product-level granularity (individual SKU analysis would be more actionable)
- Return cost estimated at flat 15% — actual costs vary by category, shipping distance, and restockability
- No customer-level repeat-return analysis

## How to Improve This Project
- Use real POS/e-commerce return data for more nuanced patterns
- Add SKU-level analysis to pinpoint specific problematic products
- Build a return probability model (logistic regression on product/customer features)
- Incorporate customer reviews/complaints as features
- Analyze return-to-repurchase conversion rates

## Production Considerations
- Monitor return rate dashboards by category and reason weekly
- Set up automated alerts when return rates exceed historical thresholds
- Feed return reason data into product listing improvement workflows
- Track return rate impact of sizing guide improvements or listing updates

## Common Mistakes
- Treating all returns equally without segmenting by reason
- Ignoring the financial dimension (return rate alone doesn't capture impact)
- Not distinguishing between controllable reasons (defective, wrong item shipped) and uncontrollable ones (changed mind)
- Analyzing returns without considering seasonality (holiday gift returns spike in January)

## Mini Challenge / Exercises
1. Which category has the highest return *cost* (not rate)?
2. Do VIP customers return for different reasons than new customers?
3. Is there a price threshold above which return rates increase significantly?
4. Create a "return risk score" for each order based on category, customer type, and price.

## Final Summary / Key Takeaways
- Return analysis requires multi-dimensional investigation: category, reason, customer, season, and financials
- High return rate ≠ highest financial impact; unit price matters
- Actionable insights come from pairing return reasons with specific categories
- Reducing "Not as Described" returns through better listings is often the highest-ROI intervention
- New customer return rates highlight the importance of accurate expectations during acquisition